Import Library

In [1]:
import pandas as pd

Loughran-Mcdonald Sentiment Extractor

In [3]:
class LMSentimentExtractor:
    def __init__(self, csv_path = None):
        self.lexicon = self._load_lexicon(csv_path)
        self.negations = {'no', 'not', 'none', 'neither', 'never', 'nobody'}
    
    def _load_lexicon(self, csv_path):
        lexicon = {'positive': set(), 'negative': set(), 'uncertain': set(), 'litigious': set()}
        df = pd.read_csv("C:/Users/tnk20/nlp/Loughran-McDonald_MasterDictionary_1993-2024.csv")
        lexicon['negative'] = set(df[df['Negative'] > 0]['Word'].str.lower())
        lexicon['positive'] = set(df[df['Positive'] > 0]['Word'].str.lower())
        lexicon['uncertain'] = set(df[df['Uncertainty'] > 0]['Word'].str.lower())
        lexicon['litigious'] = set(df[df['Litigious'] > 0]['Word'].str.lower())
        print(">> Loaded external LM Dictionary.")        

    def get_sentiment_features(self, text):
        tokens = nltk.word_tokenize(text.lower())
        total_tokens = len(tokens)
        if total_tokens == 0:
            return [0.0, 0.0, 0.0, 0.0]

        counts = {'positive': 0, 'negative': 0, 'uncertain': 0, 'litigious': 0}
            
        for i, token in enumerate(tokens):
            # Check for negation in the preceding 3 words 
            is_negated = False
            start_window = max(0, i - 3)
            if any(t in self.negations for t in tokens[start_window:i]):
                is_negated = True

            # Logic: If negated, we might flip 'positive' to 'negative' or ignore.
            # Standard LM approach is often to ignore positive words if negated.
            
            if token in self.lexicon['positive']:
                if not is_negated:
                    counts['positive'] += 1
                # If negated positive (e.g. "not good"), some implementations count as negative
                # strictly following the paper's simple "counts" logic implies we just skip the positive count.
            
            elif token in self.lexicon['negative']:
                # Negated negative (e.g. "no recession") is usually treated as neutral (ignored)
                if not is_negated:
                    counts['negative'] += 1
            
            elif token in self.lexicon['uncertain']:
                counts['uncertain'] += 1
            
            elif token in self.lexicon['litigious']:
                counts['litigious'] += 1

        # Normalize by document length 
        return [
            counts['positive'] / total_tokens,
            counts['negative'] / total_tokens,
            counts['uncertain'] / total_tokens,
            counts['litigious'] / total_tokens
        ]

Data Processor and Feature Engineering

In [4]:
class FedDataProcessor:
    def __init__(self):
        self.tfidf_vectorizer = TfidfVectorizer(
            max_features=CONFIG['tfidf_features'],
            stop_words='english',
            norm='l2'
        )
        self.lm_extractor = LMSentimentExtractor() # Pass csv_path='LoughranMcDonald_MasterDictionary.csv' here

    def preprocess_text(self, text):
        """
        Cleaning based on Section V.A [cite: 164-166]
        Note: Paper excludes lemmatization for domain-specific terms[cite: 166].
        """
        # Remove non-alpha chars, keep spaces
        text = re.sub(r'[^a-zA-Z\s]', '', text.lower())
        return text

    def process_pipeline(self, df_economic, df_text):
        print(">> Step 1: Text Preprocessing...")
        clean_texts = df_text['full_text'].apply(self.preprocess_text)

        print(">> Step 2: Generating TF-IDF Features (Method 1)...")
        # Creates T_i vector [cite: 264]
        tfidf_matrix = self.tfidf_vectorizer.fit_transform(clean_texts).toarray()
        tfidf_cols = [f'tfidf_{i}' for i in range(tfidf_matrix.shape[1])]
        df_tfidf = pd.DataFrame(tfidf_matrix, columns=tfidf_cols, index=df_text.index)

        print(">> Step 3: Generating LM Sentiment Features (Method 1)...")
        # Creates L_i vector [cite: 310]
        lm_features = df_text['full_text'].apply(self.lm_extractor.get_sentiment_features)
        df_lm = pd.DataFrame(lm_features.tolist(), 
                             columns=['lm_pos', 'lm_neg', 'lm_unc', 'lm_lit'], 
                             index=df_text.index)

        print(">> Step 4: Merging Structured and Unstructured Data...")
        # Concatenate: F_i <- [E_i, T_i, L_i] [cite: 311]
        # Align by index (Date)
        full_df = pd.concat([df_economic, df_tfidf, df_lm], axis=1).dropna()
        
        return full_df

XGBoost Model Training

In [5]:
class FedPredictor:
    def __init__(self):
        # Gradient Boosting with params from Section IV.D 
        self.model = xgb.XGBClassifier(
            n_estimators=CONFIG['n_estimators'],
            learning_rate=CONFIG['learning_rate'],
            max_depth=CONFIG['max_depth'],
            min_child_weight=CONFIG['min_samples_leaf'],
            objective='multi:softprob', # For 3-class probability
            num_class=3,
            random_state=CONFIG['random_state'],
            eval_metric='mlogloss'
        )
        self.scaler = StandardScaler()

    def train_evaluate(self, X, y):
        """
        Stratified 5-Fold CV + SMOTE [cite: 136-137]
        """
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=CONFIG['random_state'])
        auc_scores, acc_scores = [], []

        print(f">> Training XGBoost on {X.shape[1]} features (Hybrid Method 1)...")

        for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
            X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

            # Standardize Features (Paper implies standardization of economic vars)
            X_train_scaled = self.scaler.fit_transform(X_train)
            X_test_scaled = self.scaler.transform(X_test)

            # Apply SMOTE to Training Data Only [cite: 147]
            smote = SMOTE(random_state=CONFIG['random_state'])
            X_res, y_res = smote.fit_resample(X_train_scaled, y_train)

            # Train
            self.model.fit(X_res, y_res)

            # Predict
            y_pred = self.model.predict(X_test_scaled)
            y_proba = self.model.predict_proba(X_test_scaled)

            # Metrics
            # Multi-class AUC (OvR)
            auc = roc_auc_score(y_test, y_proba, multi_class='ovr')
            acc = accuracy_score(y_test, y_pred)
            
            auc_scores.append(auc)
            acc_scores.append(acc)

        print("\n=== METHOD 1 RESULTS [cite: 154] ===")
        print(f"Mean Test AUC: {np.mean(auc_scores):.4f}")
        print(f"Mean Test Accuracy: {np.mean(acc_scores):.4f}")
        return self.model

Collect Data

In [35]:
import pandas as pd
import os
import numpy as np

def process_economic_data(base_path):
    # 1. Map filenames to their specific value columns
    #    Format: 'Key': ('Filename.csv', 'Column_Header', 'Lag')
    files = {
        'HOUST':    ('HOUST.csv',    'HOUST',     12), # Year-over-Year
        'HPI':      ('HPI.csv',      'CSUSHPISA', 12), # Year-over-Year
        'CPI':      ('CPIAUCSL.csv', 'CPIAUCSL',  12), # Inflation
        'PCE':      ('PCE.csv',      'PCE',       12), # Inflation
        'FEDFUNDS': ('FEDFUNDS.csv', 'FEDFUNDS',  1),  # Inertia
        'SPREAD':   ('T10Y3MM.csv',  'T10Y3MM',   1),  # Spread
        'UMICH':    ('UMICH.csv',    'MICH',      1),  # Sentiment
        'NFP':      ('NFP.csv',      'PAYEMS',    1),  # Labor
        'BALANCE':  ('BOPGSTB.csv',  'BOPGSTB',   1),  # Trade Balance
        'UNRATE':   ('UNRATE.csv',   'UNRATE',    1),  # Unemployment
        'RSALES':   ('RSAFS.csv',    'RSAFS',     12)  # Retail Sales (NEW)
    }

    series_list = []

    # --- 2. Process Monthly Data ---
    print(">> Processing Monthly Data...")
    for key, (filename, col_name, lag) in files.items():
        file_path = os.path.join(base_path, filename)
        
        if os.path.exists(file_path):
            temp_df = pd.read_csv(file_path, parse_dates=['observation_date'])
            temp_df = temp_df.set_index('observation_date').sort_index()
            series = temp_df[col_name]
            
            # Transformation Logic
            if key == 'RSALES':
                # Explicit request: Rsales_diff_year
                transformed = series.diff(12)
                transformed.name = 'Rsales_diff_year'
            elif key in ['CPI', 'PCE']:
                transformed = series.pct_change(lag) * 100
                transformed.name = f'{key}_Inflation_Rate'
            else:
                # Default naming convention
                suffix = "year" if lag == 12 else "prev"
                transformed = series.diff(lag)
                transformed.name = f'{key}_diff_{suffix}'
            
            series_list.append(transformed)

            # Special Case: Derive 'prev_decision' from FEDFUNDS
            if key == 'FEDFUNDS':
                # 1. Calculate change
                rate_change = series.diff(1)
                # 2. Discretize: Hike (+1), Cut (-1), Hold (0)
                # Threshold 0.125 accounts for small market fluctuations vs actual 0.25 hikes
                decision_proxy = pd.cut(rate_change, 
                                      bins=[-np.inf, -0.125, 0.125, np.inf], 
                                      labels=[-1, 0, 1]).astype(float)
                # 3. Lag by 1 to get *previous* decision
                prev_decision = decision_proxy.shift(1)
                prev_decision.name = 'prev_decision'
                series_list.append(prev_decision)

        else:
            print(f"Warning: File not found {filename}")

    # --- 3. Process Quarterly GDP Data (Resample & Diff) ---
    print(">> Processing GDP Data...")
    try:
        # Load Real GDP
        gdp_path = os.path.join(base_path, "GDPC1.csv")
        if os.path.exists(gdp_path):
            gdp = pd.read_csv(gdp_path, parse_dates=['observation_date'])
            gdp = gdp.set_index('observation_date').resample('MS').ffill()
            
            # NEW: Calculate GDP_diff_year
            gdp_diff = gdp['GDPC1'].diff(12)
            gdp_diff.name = 'GDP_diff_year'
            series_list.append(gdp_diff)
        
        # Load Potential GDP for Output Gap
        pot_path = os.path.join(base_path, "GDPPOT.csv")
        if os.path.exists(gdp_path) and os.path.exists(pot_path):
            pot = pd.read_csv(pot_path, parse_dates=['observation_date'])
            pot = pot.set_index('observation_date').resample('MS').ffill()
            
            # Align and Calculate Output Gap
            gdp_combined = pd.concat([gdp['GDPC1'], pot['GDPPOT']], axis=1).dropna()
            output_gap = ((gdp_combined['GDPC1'] - gdp_combined['GDPPOT']) / gdp_combined['GDPPOT']) * 100
            output_gap.name = 'Output_Gap'
            series_list.append(output_gap)
            
    except Exception as e:
        print(f"Error processing GDP data: {e}")

    # --- 4. Merge All Data ---
    df_final = pd.concat(series_list, axis=1)

    # --- 5. Calculate Taylor Rate ---
    if 'PCE_Inflation_Rate' in df_final.columns and 'Output_Gap' in df_final.columns:
        r_star = 2.0
        pi_star = 2.0
        pi = df_final['PCE_Inflation_Rate']
        y = df_final['Output_Gap']
        df_final['Taylor_Rate'] = r_star + pi + 0.5 * (pi - pi_star) + 0.5 * y
    
    # Drop rows with NaNs (created by lagging)
    df_final = df_final.dropna()
    
    return df_final

# --- EXECUTION ---
path = "C:/Users/tnk20/nlp/structured_data/"
df_clean = process_economic_data(path)

# ... [Your existing code above] ...

# --- EXECUTION ---
# path = "C:/Users/tnk20/nlp/structured_data/"
# df_clean = process_economic_data(path)

# Option 1: Print the Index object containing column names
print("Column Index:")
print(df_clean.columns)

>> Processing Monthly Data...
>> Processing GDP Data...
Column Index:
Index(['HOUST_diff_year', 'HPI_diff_year', 'CPI_Inflation_Rate',
       'PCE_Inflation_Rate', 'FEDFUNDS_diff_prev', 'prev_decision',
       'SPREAD_diff_prev', 'UMICH_diff_prev', 'NFP_diff_prev',
       'BALANCE_diff_prev', 'UNRATE_diff_prev', 'Rsales_diff_year',
       'GDP_diff_year', 'Output_Gap', 'Taylor_Rate'],
      dtype='object')


In [37]:
# --- STANDARDIZATION ---
# 1. Create a new DataFrame to avoid overwriting the original
df_standardized = df_clean.copy()

# 2. Apply Z-score Standardization
# (Value - Mean) / Standard Deviation
for col in df_standardized.columns:
    df_standardized[col] = (df_standardized[col] - df_standardized[col].mean()) / df_standardized[col].std()

print("\nStandardized DataFrame (First 5 rows):")
print(df_standardized.head())

print("\nVerification (Mean should be ~0, Std should be 1):")
print(df_standardized.describe().loc[['mean', 'std']])


Standardized DataFrame (First 5 rows):
                  HOUST_diff_year  HPI_diff_year  CPI_Inflation_Rate  \
observation_date                                                       
2000-02-01               0.348772       0.201295            0.845761   
2000-03-01              -0.396136       0.225717            1.285847   
2000-04-01               0.374607       0.248626            0.680691   
2000-05-01              -0.094728       0.268406            0.776661   
2000-06-01               0.060282       0.280315            1.263853   

                  PCE_Inflation_Rate  FEDFUNDS_diff_prev  prev_decision  \
observation_date                                                          
2000-02-01                  2.199295            1.793941       1.959447   
2000-03-01                  2.400475            0.824433       1.959447   
2000-04-01                  1.750997            1.127404      -0.024699   
2000-05-01                  1.714433            1.612158       1.959447   
2000-

In [12]:
class EconomicDataLoader:
    def __init__(self, base_dir, file_map):
        self.base_dir = base_dir
        self.file_map = file_map

    def get_path(self, key):
        """Constructs full path from base directory and filename."""
        filename = self.file_map.get(key)
        if filename:
            return os.path.join(self.base_dir, filename)
        return None

    def load_and_process(self):
        """
        Loads CSVs, applies Paper Transformations (YoY/MoM), and merges.
        """
        dfs = []
        print(f">> Loading Economic Data from: {self.base_dir}")

        # 1. Inflation (CPI & PCE) -> Year-over-Year % Change
        [cite_start]# [cite: 76] "Inflation measures: CPI, PCE"
        for name, key in [('CPI', 'CPI'), ('PCE', 'PCE')]:
            path = self.get_path(key)
            if path and os.path.exists(path):
                df = self._read_fred_csv(path, name)
                # Calculate YoY % Change (12 month lag)
                df[f'{name}_diff_year'] = df[name].pct_change(12) * 100
                dfs.append(df[[f'{name}_diff_year']])

        # 2. Housing (HOUST & HPI) -> Year-over-Year Difference
        # [cite_start]SHAP plots show 'HOUST_diff_year' and 'HPI_diff_year' [cite: 262, 265]
        for name, key in [('HOUST', 'HOUST'), ('HPI', 'HPI')]:
            path = self.get_path(key)
            if path and os.path.exists(path):
                df = self._read_fred_csv(path, name)
                df[f'{name}_diff_year'] = df[name].diff(12)
                dfs.append(df[[f'{name}_diff_year']])

        # 3. Labor (NFP & UNRATE) -> Month-over-Month Difference
        # [cite_start]SHAP plots show 'NFP_diff_prev' and 'Unemp_diff_prev' [cite: 268, 277]
        if self.get_path('NFP'):
            df = self._read_fred_csv(self.get_path('NFP'), 'NFP')
            df['NFP_diff_prev'] = df['NFP'].diff(1)
            dfs.append(df[['NFP_diff_prev']])

        if self.get_path('UNEMP'):
            df = self._read_fred_csv(self.get_path('UNEMP'), 'UNRATE')
            df['Unemp_diff_prev'] = df['UNRATE'].diff(1)
            dfs.append(df[['Unemp_diff_prev']])

        # 4. Spread (T10Y3MM) -> Month-over-Month Difference
        # [cite_start]SHAP plot shows '10YUST_diff_prev' [cite: 258]
        if self.get_path('SPREAD'):
            df = self._read_fred_csv(self.get_path('SPREAD'), 'Spread')
            df['Spread_diff_prev'] = df['Spread'].diff(1)
            dfs.append(df[['Spread_diff_prev']])

        # Merge all
        if not dfs:
            raise ValueError("No economic data loaded! Check paths.")
            
        df_final = reduce(lambda left, right: pd.merge(left, right, on='DATE', how='outer'), dfs)
        df_final = df_final.ffill().dropna() # Handle lags
        
        print(f">> Economic Data Ready: {df_final.shape} samples.")
        return df_final

    def _read_fred_csv(self, path, col_name):
        try:
            df = pd.read_csv(path, parse_dates=['DATE'], index_col='DATE')
            df.columns = [col_name]
            return df
        except Exception as e:
            print(f"!! Warning: Could not load {path}: {e}")
            return pd.DataFrame()

In [14]:
if __name__ == "__main__":
    # 1. Load REAL Economic Data
    # Ensure your CSV files are in the same folder or update paths in CONFIG['data_files']
    econ_loader = EconomicDataLoader(CONFIG['data_files'])
    df_econ = econ_loader.load_and_process()

TypeError: EconomicDataLoader.__init__() missing 1 required positional argument: 'file_map'